# Produtividade da Gerel
#### Dados extraídos dos sistemas SAPS e Workflow.

In [1]:
import pandas as pd
import numpy as np
import datetime
from sqlalchemy import create_engine
from IPython.display import display

In [2]:
# Carrega os dados diretamente do banco SQLite gerado pelo importador
from pathlib import Path
from sqlalchemy import create_engine

DB_PATH = Path('gerel_produtividade.db')
if not DB_PATH.exists():
    raise FileNotFoundError(f"Banco de dados nao encontrado: {DB_PATH.resolve()}")

engine = create_engine(f"sqlite:///{DB_PATH.as_posix()}")

c_pessoal = pd.read_sql_table('controle_pessoa', engine)
relatorio = pd.read_sql_table('relatorio_ocorrencias', engine)
saps = pd.read_sql_table('saps_atendimentos', engine)

# O banco usa nomes padronizados com underscore. As celulas antigas do notebook usam os nomes originais dos arquivos.
c_pessoal = c_pessoal.rename(columns={
    'DATA_DE_NASCIMENTO': 'DATA DE NASCIMENTO',
    'COORDENADOR_SUPERVISOR_GERENTE': 'COORDENADOR/SUPERVISOR/GERENTE',
    'COORDENADOR_GERENTE_DIRETOR': 'COORDENADOR/GERENTE/DIRETOR',
    'HORARIO_DE_ENTRADA': 'HORARIO DE ENTRADA',
    'HORARIO_DE_SAIDA': 'HORARIO DE SAIDA',
    'DATA_DA_ADMISSAO': 'DATA DA ADMISSAO',
    'DATA_DE_EFETIVACAO': 'DATA DE EFETIVACAO',
    'CARGA_HORARIA': 'CARGA HORARIA',
    'EMAIL_DA_EMPRESA': 'EMAIL DA EMPRESA',
    'TEMPO_DE_CASA': 'TEMPO DE CASA',
    'PERFIL_INTERACT': 'PERFIL - INTERACT',
    'ENDERECO_COMPLETO': 'ENDERECO COMPLETO',
    'TELEFONE_RESIDENCIAL': 'TELEFONE RESIDENCIAL',
    'TELEFONE_CELULAR': 'TELEFONE CELULAR',
    'DATA_DESLIGAMENTO': 'DESLIGAMENTO DATA',
    'TIPO_DE_TO': 'TIPO DE TO',
    'EASY_CALL': 'EASY CALL',
    'PORTAL_SAPS_DESKTOP': 'PORTAL DO CONHECIMENTO / SAPS/ DESKTOP',
    'OS_CANCELAMENTO': 'OS (CANCELAMENTO)',
    'MAT_ANTERIOR': 'MAT ANTERIOR',
    'DATA_DA_ALTERACAO': 'DATA DA ALTERACAO',
    'PREVISAO_RETORNO': 'PREVISAO RETORNO',
    'INICIO_TREINAMENTO': 'INICIO DE TREINAMENTO',
    'UNIDADE_DE_ORIGEM': 'UNIDADE DE ORIGEM',
})

relatorio = relatorio.rename(columns={
    'Nome_Beneficiario': 'Nome do Beneficiario',
    'Tipo_Atendimento': 'Tipo de Atendimento',
    'Tipo_Macro': 'Tipo Macro',
    'Tipo_Ocorrencia': 'Tipo Ocorrencia',
    'Data_Entrada': 'Data de Entrada',
    'Hora_Entrada': 'Hora de Entrada',
    'Data_Final': 'Data Final',
    'Hora_Final': 'Hora Final',
    'Qde_NIP': 'Qde NIP',
    'Qde_Liminar': 'Qde Liminar',
    'Qde_Prioridade': 'Qde Prioridade',
    'Tipo_Prazo': 'Tipo de Prazo',
    'Prazo_Planejado': 'Prazo Planejado',
    'Prazo_Real': 'Prazo Real',
    'SLA_da_Ficha': 'SLA da Ficha',
    'Num_Etapa': 'Num Etapa',
    'Usuario_Mov': 'Usuario Mov.',
    'Setor_Usuario_Mov': 'Setor Usuario Mov.',
    'Subsetor_Usuario_Mov': 'Subsetor Usuario Mov.',
    'Setor_Etapa': 'Setor Etapa',
    'Subsetor_Etapa': 'Subsetor Etapa',
    'Data_Inicio_Etapa': 'Data Inicio Etapa',
    'Hora_Inicio_Etapa': 'Hora Inicio Etapa',
    'Data_Final_Etapa': 'Data Final Etapa',
    'Hora_Final_Etapa': 'Hora Final Etapa',
    'Status_da_Etapa': 'Status da Etapa',
    'Prazo_Planejado_Etapa': 'Prazo Planejado Etapa',
    'Prazo_Real_Etapa': 'Prazo Real Etapa',
    'SLA_Etapa': 'SLA Etapa',
})

saps = saps.rename(columns={
    'Tipo_Atendimento': 'Tipo Atendimento',
    'No_Atendimento': 'No.Atendimento',
    'Data_Entrada': 'Data de entrada',
    'Matricula': 'Matr?cula',
    'Data_Da_Proposta': 'Data da proposta',
    'Endereco': 'Endere?o',
    'Data_De_Fechamento': 'Data de fechamento',
    'Operador_Final': 'Operador Final',
    'Sub_Motivos': 'Sub Motivos',
    'Medico_Funcionario': 'M?dico|Funcion?rio',
    'Critico': 'Cr?tico',
    'Caso_Critico': 'Caso/Cr?tico',
    'Follow_Up': 'Follow up',
    'Follow_Up_Motivo': 'Follow up/Motivo',
    'Reclamacao': 'Reclama??o',
    'Codigo_Do_Solicitante': 'C?digo do Solicitante',
    'Codigo_Do_Grupo_1': 'C?digo do Grupo',
    'Grupo_1': 'Grupo',
    'Codigo_Do_Executor': 'C?digo do Executor',
    'Codigo_Do_Grupo_2': 'C?digo do Grupo.1',
    'Grupo_2': 'Grupo.1',
    'Diagnostico': 'Diagn?stico',
    'TUSS_Inicial': 'TUSS Inicial',
})

# Mantem comportamento parecido com a leitura dos arquivos: campos vazios viram NaN nas analises.
c_pessoal = c_pessoal.mask(c_pessoal == '')
relatorio = relatorio.mask(relatorio == '')
saps = saps.mask(saps == '')

print('Dados carregados do banco gerel_produtividade.db')
print(f"controle_pessoa: {len(c_pessoal):,} linhas")
print(f"relatorio_ocorrencias: {len(relatorio):,} linhas")
print(f"saps_atendimentos: {len(saps):,} linhas")


Dados carregados do banco gerel_produtividade.db
controle_pessoa: 396 linhas
relatorio_ocorrencias: 196,984 linhas
saps_atendimentos: 2,643,974 linhas


In [3]:
print("Colunas originais encontradas:", saps.columns.tolist())

Colunas originais encontradas: ['Tipo Atendimento', 'No.Atendimento', 'Operador', 'Data de entrada', 'Matr?cula', 'Nome', 'Data da proposta', 'Email', 'Telefone', 'Endere?o', 'Bairro', 'Cidade', 'Status', 'Data de fechamento', 'Operador Final', 'RegistroANS', 'Motivo', 'Prestador', 'Motivos', 'Sub Motivos', 'M?dico|Funcion?rio', 'CEP', 'Cr?tico', 'Local', 'Caso/Cr?tico', 'Follow up', 'Follow up/Motivo', 'Reclama??o', 'Senha', 'C?digo do Solicitante', 'Solicitante', 'CRM', 'C?digo do Grupo', 'Grupo', 'Especialidade', 'C?digo do Executor', 'Executor', 'C?digo do Grupo.1', 'Grupo.1', 'CID', 'Diagn?stico', 'TUSS Inicial']


In [4]:
saps.columns = saps.columns.str.strip()

In [5]:
print("=== OLHE AQUI NO TERMINAL OS NOMES REAIS DAS SUAS COLUNAS ===")
print(saps.columns.tolist())
print("=============================================================")

=== OLHE AQUI NO TERMINAL OS NOMES REAIS DAS SUAS COLUNAS ===
['Tipo Atendimento', 'No.Atendimento', 'Operador', 'Data de entrada', 'Matr?cula', 'Nome', 'Data da proposta', 'Email', 'Telefone', 'Endere?o', 'Bairro', 'Cidade', 'Status', 'Data de fechamento', 'Operador Final', 'RegistroANS', 'Motivo', 'Prestador', 'Motivos', 'Sub Motivos', 'M?dico|Funcion?rio', 'CEP', 'Cr?tico', 'Local', 'Caso/Cr?tico', 'Follow up', 'Follow up/Motivo', 'Reclama??o', 'Senha', 'C?digo do Solicitante', 'Solicitante', 'CRM', 'C?digo do Grupo', 'Grupo', 'Especialidade', 'C?digo do Executor', 'Executor', 'C?digo do Grupo.1', 'Grupo.1', 'CID', 'Diagn?stico', 'TUSS Inicial']


In [6]:
relatorio['Data de Entrada'] = pd.to_datetime(relatorio['Data de Entrada'], dayfirst=True, errors='coerce')
relatorio['MES'] = relatorio['Data de Entrada'].dt.strftime('%y-%m')


In [7]:
#saps.head(1).T


In [8]:
# O método dropna varre a tabela procurando por dados nulos (NaN)
# O parâmetro axis=1 avisa ao motor que estamos avaliando as colunas, e não as linhas
# O parâmetro how='all' é a trava de segurança: ele só deleta a coluna se 100% das linhas dela estiverem vazias
#saps = saps.dropna(axis=1, how='all')

# Opcional: Imprime as informações atualizadas da base para você auditar o quanto de memória e colunas nós economizamos
#saps.info()

In [9]:
# Remove espa?os em branco invis?veis que podem ter vindo na exporta??o para garantir o match perfeito
saps.columns = saps.columns.str.strip()

# Coluna de data de entrada do SAPS carregada do banco SQLite
coluna_data = 'Data de entrada'

# O banco pode conter datas em formato brasileiro ou ISO; o pandas detecta ambos com errors='coerce'
saps[coluna_data] = pd.to_datetime(saps[coluna_data], dayfirst=True, errors='coerce')

# Cria a nova coluna 'MES' extraindo o Ano-M?s (ex: 26-05) para permitir agrupamentos futuros
saps['MES'] = saps[coluna_data].dt.strftime('%y-%m')

#display(saps[[coluna_data, 'MES']].head())


In [10]:
produtividade = pd.crosstab(index=[relatorio['Usuario Mov.']], columns=relatorio['MES'], margins=True, margins_name='TOTAL')
produtividade = pd.concat([produtividade.drop('TOTAL').sort_values(by='TOTAL', ascending=False),produtividade.loc[['TOTAL']]])
#display(produtividade)

In [11]:
demanda_mais_antiga = relatorio[relatorio['Status'] == 'PENDENTE'].sort_values(by='Data de Entrada', ascending=True).head(1).T
#display(demanda_mais_antiga)

In [12]:
logins = c_pessoal['LOGIN'].tolist()
rel_filtrado = relatorio[relatorio['Usuario Mov.'].isin(logins)]

produtividade_filtrada = pd.crosstab(
    index=rel_filtrado['Usuario Mov.'],
    columns=rel_filtrado['MES'],
    margins=True,
    margins_name='TOTAL'
)

produtividade_filtrada = pd.concat([
    produtividade_filtrada.drop('TOTAL').sort_values(by='TOTAL', ascending=False),
    produtividade_filtrada.loc[['TOTAL']]
])

#display(produtividade_filtrada)

In [13]:
info_pessoal = c_pessoal[['LOGIN', 'STATUS', 'TIPO', 'COORDENADOR/SUPERVISOR/GERENTE']].rename(columns={'STATUS': 'Status', 'TIPO': 'Tipo', 'COORDENADOR/SUPERVISOR/GERENTE': 'Supervisor'})
produtividade_filtrada = produtividade_filtrada.reset_index()
produtividade_filtrada = pd.merge(produtividade_filtrada, info_pessoal, left_on='Usuario Mov.', right_on='LOGIN', how='left')
produtividade_filtrada = produtividade_filtrada.drop('LOGIN', axis=1)
colunas_mes = [c for c in produtividade_filtrada.columns if c not in ['Usuario Mov.', 'Status', 'Tipo', 'Supervisor', 'TOTAL']]
produtividade_filtrada = produtividade_filtrada[['Usuario Mov.', 'Status', 'Tipo', 'Supervisor'] + colunas_mes + ['TOTAL']]
#display(produtividade_filtrada)

In [14]:
# Cria a matriz cruzando os operadores do SAP com os meses e gera as totalizações automáticas nas margens
produtividade_saps = pd.crosstab(index=saps['Operador'], columns=saps['MES'], margins=True, margins_name='TOTAL')

# Aplica a nossa técnica de slicing (.iloc) para separar o corpo da tabela, ordenar a coluna 'TOTAL' em ordem decrescente, e colar a linha de total de volta no fundo
produtividade_saps = pd.concat([produtividade_saps.iloc[:-1].sort_values(by='TOTAL', ascending=False), produtividade_saps.iloc[[-1]]])
#display(produtividade_saps)

In [15]:
supervisores_unicos = produtividade_filtrada['Supervisor'].dropna().unique()

colunas_mes = [c for c in produtividade_filtrada.columns 
               if c not in ['Usuario Mov.', 'Status', 'Tipo', 'Supervisor', 'TOTAL']]

for supervisor in supervisores_unicos:
    equipe = produtividade_filtrada[
        produtividade_filtrada['Supervisor'] == supervisor
    ].copy()

    # Adicionar linha de total da equipe
    total_equipe = equipe[colunas_mes + ['TOTAL']].sum()
    total_equipe['Usuario Mov.'] = 'TOTAL EQUIPE'
    total_equipe['Status'] = ''
    total_equipe['Tipo'] = ''
    total_equipe['Supervisor'] = ''

    equipe = pd.concat([
        equipe,
        pd.DataFrame([total_equipe])
    ], ignore_index=True)

    print(f"\n====== EQUIPE: {supervisor} ======")
   # display(equipe.style.hide(axis='index'))


====== EQUIPE: EDINEIDE SILVA ======

====== EQUIPE: MARINA OLIVEIRA ======

====== EQUIPE: BRENDO SANTOS ======

====== EQUIPE: ADRIANA NOBRE ======

====== EQUIPE: GESTOR AFASTAMENTO ======

====== EQUIPE: DAIANE MACEDO ======

====== EQUIPE: ALAN VIEIRA ======

====== EQUIPE: LEONARDO RIBEIRO ======

====== EQUIPE: ANA PAULA MATTOS ======

====== EQUIPE: ANDRESSA MEDEIROS ======

====== EQUIPE: MARLON PANISSET ======

====== EQUIPE: ELISANGELA BORGES ======

====== EQUIPE: CRISTIANE MACEDO ======

====== EQUIPE: DANIELE DANTAS ======

====== EQUIPE: GILENO XAVIER ======

====== EQUIPE: ALINE AZEVEDO ======

====== EQUIPE: PRISCILA LAGE ======

====== EQUIPE: RAFAEL COSTA ======

====== EQUIPE: ROBERT FERRAZ ======

====== EQUIPE: SOLANGE TEIXEIRA ======

====== EQUIPE: ROSANGELA CAETANO ======

====== EQUIPE: SIMONE CASTRO ======

====== EQUIPE: ELAINE MOURA ======

====== EQUIPE: LAIS COSTA ======


In [16]:
op1 = saps[['Operador', 'MES']].rename(columns={'Operador': 'login'})
op2 = saps[['Operador Final', 'MES']].rename(columns={'Operador Final': 'login'})

In [17]:
saps_long = pd.concat([
    saps[['Operador', 'MES']].assign(idx=saps.index).rename(columns={'Operador': 'login'}),
    saps[['Operador Final', 'MES']].assign(idx=saps.index).rename(columns={'Operador Final': 'login'})
])

In [18]:
saps_long = saps_long.dropna(subset=['login'])
saps_long['login'] = saps_long['login'].str.strip()

In [19]:
saps_long = saps_long.drop_duplicates(subset=['idx', 'login'])

In [20]:
saps_filtrado = saps_long[saps_long['login'].isin(logins)]

In [21]:
saps_filtrado = saps_filtrado.reset_index(drop=True)

prod_saps = pd.crosstab(
    index=saps_filtrado['login'],
    columns=saps_filtrado['MES'],
    margins=True,
    margins_name='TOTAL'
)

prod_saps = pd.concat([
    prod_saps.drop('TOTAL').sort_values(by='TOTAL', ascending=False),
    prod_saps.loc[['TOTAL']]
])

In [22]:
prod_saps = prod_saps.reset_index().rename(columns={'login': 'Usuario Mov.'})

prod_saps = pd.merge(prod_saps, info_pessoal, left_on='Usuario Mov.', right_on='LOGIN', how='left')
prod_saps = prod_saps.drop('LOGIN', axis=1)

colunas_mes_saps = [c for c in prod_saps.columns 
                    if c not in ['Usuario Mov.', 'Status', 'Tipo', 'Supervisor', 'TOTAL']]

prod_saps = prod_saps[['Usuario Mov.', 'Status', 'Tipo', 'Supervisor'] + colunas_mes_saps + ['TOTAL']]

In [23]:
supervisores_unicos_saps = prod_saps['Supervisor'].dropna().unique()

for supervisor in supervisores_unicos_saps:
    equipe = prod_saps[prod_saps['Supervisor'] == supervisor].copy()

    total_equipe = equipe[colunas_mes_saps + ['TOTAL']].sum()
    total_equipe['Usuario Mov.'] = 'TOTAL EQUIPE'
    total_equipe['Status'] = ''
    total_equipe['Tipo'] = ''
    total_equipe['Supervisor'] = ''

    equipe = pd.concat([
        equipe,
        pd.DataFrame([total_equipe])
    ], ignore_index=True)

    print(f"\n====== EQUIPE SAPS: {supervisor} ======")
   # display(equipe.style.hide(axis='index'))



====== EQUIPE SAPS: MARINA OLIVEIRA ======

====== EQUIPE SAPS: EDINEIDE SILVA ======

====== EQUIPE SAPS: ELISANGELA BORGES ======

====== EQUIPE SAPS: ALAN VIEIRA ======

====== EQUIPE SAPS: ANA PAULA MATTOS ======

====== EQUIPE SAPS: DAIANE MACEDO ======

====== EQUIPE SAPS: MARLON PANISSET ======

====== EQUIPE SAPS: ANDRESSA MEDEIROS ======

====== EQUIPE SAPS: CRISTIANE MACEDO ======

====== EQUIPE SAPS: LEONARDO RIBEIRO ======

====== EQUIPE SAPS: GILENO XAVIER ======

====== EQUIPE SAPS: PRISCILA LAGE ======

====== EQUIPE SAPS: BRENDO SANTOS ======

====== EQUIPE SAPS: RAFAEL COSTA ======

====== EQUIPE SAPS: GESTOR AFASTAMENTO ======

====== EQUIPE SAPS: DANIELE DANTAS ======

====== EQUIPE SAPS: ALINE AZEVEDO ======

====== EQUIPE SAPS: ROSANGELA CAETANO ======

====== EQUIPE SAPS: SIMONE CASTRO ======

====== EQUIPE SAPS: SOLANGE TEIXEIRA ======

====== EQUIPE SAPS: ROBERT FERRAZ ======

====== EQUIPE SAPS: LAIS COSTA ======

====== EQUIPE SAPS: ELAINE MOURA ======


In [24]:
# Passo 1: Empilha as duas tabelas verticalmente (Workflow + SAPS) ignorando as linhas antigas de TOTAL
base_bruta_unificada = pd.concat([
    produtividade_filtrada[produtividade_filtrada['Usuario Mov.'] != 'TOTAL'],
    prod_saps[prod_saps['Usuario Mov.'] != 'TOTAL']
], ignore_index=True)

# Passo 2: Funde as listas de meses usando 'set' para remover duplicatas e ordena cronologicamente
colunas_mes_unificadas = sorted(list(set(colunas_mes + colunas_mes_saps)))
colunas_numericas_unificadas = colunas_mes_unificadas + ['TOTAL']

# Preenche com 0 qualquer mes que tenha ficado nulo e garante numeros inteiros no painel
base_bruta_unificada[colunas_numericas_unificadas] = (
    base_bruta_unificada[colunas_numericas_unificadas]
    .apply(pd.to_numeric, errors='coerce')
    .fillna(0)
    .astype(int)
)

# Passo 3: Agrupa os dados pela identidade do funcionario e soma os resultados numericos dos dois sistemas numa linha so
prod_unificada = base_bruta_unificada.groupby(
    ['Usuario Mov.', 'Status', 'Tipo', 'Supervisor'], as_index=False
)[colunas_numericas_unificadas].sum()
prod_unificada[colunas_numericas_unificadas] = prod_unificada[colunas_numericas_unificadas].astype(int)

# Passo 4: Laco de repeticao isolando as equipes
supervisores_unicos = prod_unificada['Supervisor'].dropna().unique()

for supervisor in supervisores_unicos:
    equipe = prod_unificada[prod_unificada['Supervisor'] == supervisor].copy()
    equipe = equipe.sort_values(by='TOTAL', ascending=False)

    total_equipe = equipe[colunas_numericas_unificadas].sum().astype(int)
    total_equipe['Usuario Mov.'] = 'TOTAL EQUIPE'
    total_equipe['Status'] = ''
    total_equipe['Tipo'] = ''
    total_equipe['Supervisor'] = ''

    equipe = pd.concat([equipe, pd.DataFrame([total_equipe])], ignore_index=True)
    equipe[colunas_numericas_unificadas] = equipe[colunas_numericas_unificadas].astype(int)

    print(f"\n====== EQUIPE UNIFICADA (WORKFLOW + SAPS): {supervisor} ======")
    display(equipe.style.hide(axis='index').format({col: '{:d}' for col in colunas_numericas_unificadas}))



====== EQUIPE UNIFICADA (WORKFLOW + SAPS): LEONARDO RIBEIRO ======


Usuario Mov.,Status,Tipo,Supervisor,25-11,25-12,26-01,26-02,26-03,26-04,26-05,TOTAL
ANADIAS,ATIVO,EFETIVO,LEONARDO RIBEIRO,0,30,384,264,351,227,66,1322
EVELYNG,ATIVO,EFETIVO,LEONARDO RIBEIRO,20,27,269,208,230,233,50,1037
ROSANEN,ATIVO,EFETIVO,LEONARDO RIBEIRO,22,31,232,165,275,205,53,983
DENISER,ATIVO,EFETIVO,LEONARDO RIBEIRO,28,59,206,122,236,194,91,936
ADRIANAF,ATIVO,EFETIVO,LEONARDO RIBEIRO,34,50,212,149,160,207,98,910
MARTINSS,ATIVO,EFETIVO,LEONARDO RIBEIRO,20,53,223,126,242,190,31,885
GIMARIA,ATIVO,EFETIVO,LEONARDO RIBEIRO,29,39,216,115,189,186,86,860
ANDREACN,ATIVO,EFETIVO,LEONARDO RIBEIRO,0,45,222,128,231,199,23,848
CLEBER,ATIVO,EFETIVO,LEONARDO RIBEIRO,34,49,186,122,202,165,33,791
HERICA,ATIVO,EFETIVO,LEONARDO RIBEIRO,30,58,83,121,206,177,94,769



====== EQUIPE UNIFICADA (WORKFLOW + SAPS): ANA PAULA MATTOS ======


Usuario Mov.,Status,Tipo,Supervisor,25-11,25-12,26-01,26-02,26-03,26-04,26-05,TOTAL
JCRIST,ATIVO,EFETIVO,ANA PAULA MATTOS,20,41,490,453,447,491,128,2070
CRISMAR,ATIVO,EFETIVO,ANA PAULA MATTOS,0,24,318,285,355,306,100,1388
JOSIANEL,FERIAS,EFETIVO,ANA PAULA MATTOS,17,16,304,314,271,353,28,1303
ADRIANAO,FERIAS,EFETIVO,ANA PAULA MATTOS,23,37,288,244,367,301,29,1289
GLAUCI,ATIVO,EFETIVO,ANA PAULA MATTOS,27,63,309,214,302,278,72,1265
FVENTURA,FERIAS,EFETIVO,ANA PAULA MATTOS,18,41,330,324,366,164,0,1243
DAVIDM,ATIVO,EFETIVO,ANA PAULA MATTOS,14,24,337,266,337,199,47,1224
WAGNER,ATIVO,EFETIVO,ANA PAULA MATTOS,0,22,269,235,325,262,48,1161
ALEPP,ATIVO,EFETIVO,ANA PAULA MATTOS,35,2,292,191,293,218,84,1115
LBARA,FERIAS,EFETIVO,ANA PAULA MATTOS,15,38,227,243,297,247,37,1104



====== EQUIPE UNIFICADA (WORKFLOW + SAPS): ALAN VIEIRA ======


...

Usuario Mov.,Status,Tipo,Supervisor,25-11,25-12,26-01,26-02,26-03,26-04,26-05,TOTAL
ECARDOSO,ATIVO,EFETIVO,LAIS COSTA,2,1,6,5,4,0,0,18
ROBERT,ATIVO,EFETIVO,LAIS COSTA,0,0,0,0,1,0,0,1
TOTAL EQUIPE,,,,2,1,6,5,5,0,0,19



====== EQUIPE UNIFICADA (WORKFLOW + SAPS): ROSANGELA CAETANO ======


Usuario Mov.,Status,Tipo,Supervisor,25-11,25-12,26-01,26-02,26-03,26-04,26-05,TOTAL
GRASSO,ATIVO,EFETIVO,ROSANGELA CAETANO,6,0,103,79,91,65,16,360
GQUEIROZ,ATIVO,EFETIVO,ROSANGELA CAETANO,9,10,0,72,123,74,44,332
TOTAL EQUIPE,,,,15,10,103,151,214,139,60,692



====== EQUIPE UNIFICADA (WORKFLOW + SAPS): ADRIANA NOBRE ======


Usuario Mov.,Status,Tipo,Supervisor,25-11,25-12,26-01,26-02,26-03,26-04,26-05,TOTAL
SUZANE,EMPRESTADO,EFETIVO,ADRIANA NOBRE,185,129,268,432,326,70,0,1410
TOTAL EQUIPE,,,,185,129,268,432,326,70,0,1410
